In [1]:
!pip install scikit-learn pandas numpy joblib matplotlib --quiet

import pandas as pd
import numpy as np
import gc
import joblib
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score

print("Imports successful")

Imports successful


In [2]:
data         = joblib.load("/kaggle/input/datasets/pes2ug23cs197/svm-splits/splits.pkl")
X_train      = data["X_train"]
X_val        = data["X_val"]
X_test       = data["X_test"]
y_train      = data["y_train"]
y_val        = data["y_val"]
y_test       = data["y_test"]
del data
gc.collect()

le           = joblib.load("/kaggle/input/datasets/pes2ug23cs197/svm-splits/label_encoder.pkl")
feature_cols = joblib.load("/kaggle/input/datasets/pes2ug23cs197/svm-splits/feature_cols.pkl")
num_classes  = len(le.classes_)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Classes: {num_classes} | Features: {X_train.shape[1]}")

Train: 34999 | Val: 7500 | Test: 7500
Classes: 1117 | Features: 1756


In [3]:
svm_base = LinearSVC(
    max_iter     = 3000,
    class_weight = "balanced",
    random_state = 42,
    C            = 0.5           
)

svm_model = CalibratedClassifierCV(
    estimator = svm_base,
    cv        = 5
)

print("Training SVM ...")
svm_model.fit(X_train, y_train)   
joblib.dump(svm_model, "svm_model.pkl")
print("svm_model.pkl saved")

Training SVM ...
svm_model.pkl saved


In [4]:
y_val_pred = svm_model.predict(X_val)
val_acc    = accuracy_score(y_val, y_val_pred)
train_acc  = accuracy_score(y_train, svm_model.predict(X_train))
gap        = train_acc - val_acc

print(f"Train Accuracy : {train_acc*100:.2f}%")
print(f"Val   Accuracy : {val_acc*100:.2f}%")
print(f"Gap            : {gap*100:.2f}%  {'- OK' if gap < 0.10 else '- Check regularization'}")

Train Accuracy : 85.77%
Val   Accuracy : 81.85%
Gap            : 3.91%  - OK


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    svm_base, X_train, y_train,
    cv      = skf,
    scoring = "accuracy",
    n_jobs  = -1
)

print(f"CV Scores : {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean      : {cv_scores.mean()*100:.2f}%")
print(f"Std Dev   : {cv_scores.std()*100:.2f}%  {'- Stable' if cv_scores.std() < 0.02 else '- High variance'}")

CV Scores : ['0.8237', '0.8229', '0.8177', '0.8173', '0.8317']
Mean      : 82.27%
Std Dev   : 0.52%  - Stable
